[![Colab Badge](https://img.shields.io/badge/Open_in_Colab-blue?style=for-the-badge)][colab-link]
<a href="javascript:void(0);" onclick="openJupyterWidget('https://github.com/nmfs-opensci/EDMW-EarthData-Workshop-2025/blob/main/tutorials/test.ipynb');">
    <img src="https://img.shields.io/badge/Open_in_JupyterHub-orange?style=for-the-badge" alt="JupyterHub Badge">
</a> [![Download Badge](https://img.shields.io/badge/Download-grey?style=for-the-badge)][download-link]

[download-link]: https://github.com/nmfs-opensci/EDMW-EarthData-Workshop-2025/blob/main/tutorials/Tutorial_3_moana-erddap.ipynb
[colab-link]: https://colab.research.google.com/github/nmfs-opensci/EDMW-EarthData-Workshop-2025/blob/main/tutorials/Tutorial_3_moana-erddap.ipynb
[jupyter-link]: https://nmfs-openscapes.2i2c.cloud/hub/user-redirect/lab?fromURL=https://raw.githubusercontent.com/nmfs-opensci/EDMW-EarthData-Workshop-2025/main/tutorials/Tutorial_3_moana-erddap.ipynb

>📘 Learning Objectives
>
> 1. Create pretty maps of the three phytoplankton classes in MOANA
> 2. Plot how all three of these phytoplankton classes change in relation to latitude
> 3. Plot how all three of these phytoplankton classes change over the course of 1 year
> 4. Create a ternary plot and map that shows where each phytoplankton class dominates


## You will need 3.7Gb for this tutorial

Go to File > Hub Control Panel > Stop my server

Then restart after chosing 3.7 Gb in the drop-down.

## Interacting with multiple phytoplankton groups simultaneously

MOANA is the first phytoplankton community composition algorithm to be released by PACE. This product returns near-surface concentrations (cells mL-1) of three different picophytoplankton (i.e., phytoplankton <2 μm in size): Prochlorococcus, Synechococcus, and autotrophic picoeukaryotes (Figure 15). The algorithm uses empirical relationships between measured cell concentrations, in situ hyperspectral remote sensing reflectances, and sea surface temperatures. Details of this algorithm can be found in [Lange et al. (2020)](https://doi.org/10.1364/OE.398127).

Picophytoplankton are composed of the cyanobacteria Prochlorococcus (∼0.8 µm) and Synechococcus (∼1 µm), as well as picoeukaryotes, which combined are responsible for 50 to 90% of all primary production in open ocean ecosystems and contribute up to 30% of carbon export in these regions. Geographically, Prochlorococcus tends to inhabit warmer and mostly oligotrophic waters surrounded by Synechococcus patches along frontal boundaries. These fronts often reside at boundaries where phytoplankton communities start to transition to higher concentrations of larger eukaryotic cells, such as picoeukaryotes and nanoeukaryotic flagellates. Thus, identification of Prochlorococcus and Synechococcus distributions may be useful in identifying trophic boundaries in oceanic ecosystems, in addition to providing insight into productivity, food web regimes, and carbon export.

Note, for now, the MOANA product is only avaialble for the Atlantic Ocean.

## *Are you ready?*

## Lock and load the data

In [1]:
# ---- Load Libraries ----
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ---- Load Data ----
url = "https://cwcgom.aoml.noaa.gov/erddap/griddap/noaa_aoml_a113_ab04_933f"
ds = xr.open_dataset(url)

# --- Get first week of Sept 2024
dataset = ds.sel(time="2024-09-01")

# Assign core variables
latitude = dataset["latitude"]
longitude = dataset["longitude"]
syn_da = dataset["syncoccus_moana"]
pico_da = dataset["picoeuk_moana"]
pro_da = dataset["prococcus_moana"]

In [2]:
print(f"Size in TB: {dataset.nbytes / 1e9:.2f} GB")
dataset.sizes

Size in TB: 0.11 GB


Frozen({'latitude': 3360, 'longitude': 2640})

In [2]:
dataset

<xarray.Dataset> Size: 106MB
Dimensions:          (latitude: 3360, longitude: 2640)
Coordinates:
    time             datetime64[ns] 8B 2024-09-01
  * latitude         (latitude) float32 13kB 69.98 69.94 69.9 ... -69.94 -69.98
  * longitude        (longitude) float32 11kB -84.98 -84.94 ... 24.94 24.98
Data variables:
    prococcus_moana  (latitude, longitude) float32 35MB ...
    syncoccus_moana  (latitude, longitude) float32 35MB ...
    picoeuk_moana    (latitude, longitude) float32 35MB ...
Attributes: (12/71)
    _lastModified:                     2025-02-28T17:56:21.000Z
    cdm_data_type:                     Grid
    comment:                           This dataset is redistributed by Atlan...
    Conventions:                       CF-1.10, ACDD-1.3, COARDS
    creator_email:                     data@oceancolor.gsfc.nasa.gov
    creator_name:                      NASA/GSFC/OBPG
    ...                                ...
    temporal_range:                    7-day
    time_coverage_end:                 2025-03-02T00:00:00Z
    time_coverage_start:               2024-04-18T00:00:00Z
    title:                             OCI L3 SMI, PACE MOANA 8DAY; via Atlan...
    Westernmost_Easting:               -84.97916
    westernmost_longitude:             -85.0

In [4]:
# let's load the data into memory since it is not so big
dataset = dataset.load()

## It's map time again

We've got three different phytoplankton classes in this MOANA data set. Let's create some pretty maps and get a sense of their relative distributions.

In [ ]:
# Define colormaps
custom_cmaps = {
    "Prochlorococcus": plt.cm.Blues,
    "Synechococcus": plt.cm.Reds,
    "Picoeukaryotes": plt.cm.Greens
}

# Label text for colorbars
colorbar_labels = {
    "Prochlorococcus": "Prochlorococcus conc. (cells mL⁻¹)",
    "Synechococcus": "Synechococcus conc. (cells mL⁻¹)",
    "Picoeukaryotes": "Picoeukaryote conc. (cells mL⁻¹)"
}

# Set up figure and axes
fig, axs = plt.subplots(1, 3, figsize=(18, 6),
                        subplot_kw={'projection': ccrs.PlateCarree()})

phyto_list = [
    ("Prochlorococcus", pro_da, axs[0]),
    ("Synechococcus", syn_da, axs[1]),
    ("Picoeukaryotes", pico_da, axs[2])
]

for title, data, ax in phyto_list:
    ax.set_title(title)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    gl = ax.gridlines(draw_labels=True)
    gl.right_labels = gl.top_labels = False

    # Plot with custom ternary-style colormap and auto robust scaling
    cmap = custom_cmaps[title]
    img = data.plot(
        ax=ax,
        x="longitude", y="latitude",
        cmap=cmap,
        robust=True,
        add_colorbar=False
    )

    # Add custom colorbar label
    cbar = plt.colorbar(img, ax=ax, orientation='horizontal', pad=0.05, shrink=0.9)
    cbar.set_label(colorbar_labels[title])

plt.tight_layout()
plt.show()

Looking good! Here you can see that little ole Prochlorococcus really runs amok in the offshore waters, where both Synechococcus and Picoeukaryotes tend to dominate at higher latitudes, and Picoeukaryotes assert their dominance in along the coastal margins... you know, to the degree that a free-floating cell can "assert" anything.

## Let's get a bit more quantitative

Let's get a little better sense of how these phytoplakton classes are geographically distributed. What we're going to do next is plot how all three of these phytoplankton classes change in relation to latitude. To do this, we'll be taking "slices" of data and averaging them. Conceptually, think of a horizontal line across 50 degrees North - we'll extract all that data, take an average, and then move on to the pixel below (e.g. 49.98 degrees North) and do the same until we hit 50 degrees South. Doing this for each phytoplankton class, we can now distill this information into something that can be put onto a line plot. 